# 🎬 Problem 2: Automated Bengali Subtitle & Closed-Caption Pipeline

**Track: Speech & Language / Media Localisation**

End-to-end pipeline: Bengali video → broadcast-quality CC (speaker-attributed, with SED events) + EN/HI subtitle tracks + QC report with ranked human review queue.

### ASR Engine (from Problem 1 — loaded in priority order)
| Priority | Model | Condition |
|---|---|---|
| 1 | **Path B PolyWhisper MoLE** | B2 + B3 + B4 completed |
| 2 | **Path A merged LoRA** | Notebooks 1–5 completed |
| 3 | Base SayedShaun/bengali-whisper-medium | Fallback |

### The 3 Pillars Must Get Right
1. **Recognition** — Bengali with inline code-switched English (e.g. *"ওর office-এ একটা meeting আছে"*)
2. **Attribution** — Stable speaker IDs across full runtime (zero identity swaps)
3. **Presentation** — CPS ≤ 17.5, CPL ≤ 38, max 2 lines, no shot straddles

### 🛡️ Toughest Test Defense
> Multi-signal Anti-Hallucination Guard catches phantom Whisper text over silence/music using VAD energy, spectral flatness, compression ratio, and Whisper no-speech probabilities. All suspects go into the **Ranked Review Queue**.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — Environment Setup & Drive Mount
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, sys, subprocess

print('⚙️  Installing dependencies...')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40.0', 'torchaudio>=2.2.0', 'soundfile>=0.12.1',
    'librosa>=0.10.1', 'opencv-python>=4.8.0', 'accelerate>=0.29.0',
    'peft>=0.10.0,<0.14.0', 'sentencepiece>=0.2.0', 'sacremoses',
    'panns-inference>=0.1.0',
    'speechbrain>=1.0.0',
])

# Colab peft/torchao probe fix
try:
    import peft.import_utils as _piu
    _piu.is_torchao_available = lambda: False
except Exception:
    pass

# ffmpeg (needed for video audio extraction)
subprocess.run(['apt-get', 'install', '-qq', 'ffmpeg'], capture_output=True)

# Mount Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/hoichoi'
except Exception:
    DRIVE_ROOT = '.'

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Environment ready  |  Device: {DEVICE}  |  Drive root: {DRIVE_ROOT}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — Clone repo & set Python path (if needed)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
REPO_DIR = os.path.join(DRIVE_ROOT, 'hoichoi')   # adjust if repo is cloned elsewhere
if not os.path.isdir(REPO_DIR):
    # Fallback: try the drive root itself (user might have the repo there directly)
    REPO_DIR = DRIVE_ROOT

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Verify the src package is importable
try:
    import problem_2_caption_diarization.src as p2src
    print(f'✅ problem_2_caption_diarization.src found at: {p2src.__file__}')
except ModuleNotFoundError:
    print('⚠️  Package not found on sys.path. Falling back to direct clone.')
    subprocess.run(['git', 'clone', 'https://github.com/your-org/hoichoi', REPO_DIR], check=False)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — Problem 1 Model Readiness Check
# Shows which ASR mode will be used based on available adapters
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BN_EXPERT      = DRIVE_ROOT + '/adapters_path_b/bengali_expert/best'
EN_EXPERT      = DRIVE_ROOT + '/adapters_path_b/english_expert/best'
ROUTER_WEIGHTS = DRIVE_ROOT + '/router_path_b/best/router.pt'
MERGED_PATH_A  = DRIVE_ROOT + '/adapters/merged'

bn_ok = os.path.exists(BN_EXPERT + '/adapter_config.json')
en_ok = os.path.exists(EN_EXPERT + '/adapter_config.json')
rt_ok = os.path.exists(ROUTER_WEIGHTS)
pa_ok = os.path.exists(MERGED_PATH_A + '/adapter_config.json')

print('=' * 60)
print('  PROBLEM 1 → PROBLEM 2  ASR Engine Readiness')
print('=' * 60)
print(f'  Path B Bengali Expert LoRA : {"✅ Found" if bn_ok else "❌ Missing — run B2"}')
print(f'  Path B English Expert LoRA : {"✅ Found" if en_ok else "❌ Missing — run B3"}')
print(f'  Path B MoLE Router         : {"✅ Found" if rt_ok else "⚠️  Missing (optional) — run B4"}')
print(f'  Path A Merged Adapter      : {"✅ Found" if pa_ok else "⚠️  Missing — run Notebooks 1-5"}')

if bn_ok and en_ok:
    asr_mode_expected = 'mole (Path B PolyWhisper) 🏆'
elif pa_ok:
    asr_mode_expected = 'path_a (merged LoRA adapter)'
else:
    asr_mode_expected = 'base (SayedShaun/bengali-whisper-medium)'

print(f'\n  Expected ASR mode  → {asr_mode_expected}')
print('=' * 60)

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 4 — Video Discovery
# Searches Drive for Hoichoi clips (.mp4 / .mkv)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import glob

search_dirs = [DRIVE_ROOT, '/content', DRIVE_ROOT + '/videos', DRIVE_ROOT + '/data']
video_files = []
for d in search_dirs:
    if os.path.isdir(d):
        video_files += glob.glob(d + '/*.mp4') + glob.glob(d + '/*.mkv')

video_files = sorted(set(video_files))

if not video_files:
    raise FileNotFoundError(
        'No video clips found!\n'
        f'Upload an .mp4 to {DRIVE_ROOT} or /content.'
    )

print(f'Found {len(video_files)} video file(s):')
for i, v in enumerate(video_files):
    sz = os.path.getsize(v) / (1024 * 1024)
    print(f'  [{i+1}] {os.path.basename(v)}  ({sz:.1f} MB)')

# Priority: feluda > first file
TARGET_VIDEO = video_files[0]
for v in video_files:
    if 'feluda' in v.lower():
        TARGET_VIDEO = v
        break

print(f'\n🎯 Target: {TARGET_VIDEO}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 5 — Run Full Pipeline
#
# BengaliSubtitlePipeline.run() handles every stage:
#   1. Audio extraction + shot cut detection
#   2. VAD + SED + Anti-Hallucination Guard
#   3. Speaker diarization (pyannote OR acoustic clustering)
#   4. ASR via Problem 1 trained model (MoLE / Path A / base)
#   5. Broadcast timed-text formatting
#   6. BN -> EN (opus-mt-bn-en) & BN -> HI (nllb-200) translation
#   7. QC audit + ranked review queue
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from problem_2_caption_diarization.src.pipeline import BengaliSubtitlePipeline

OUTPUT_DIR = os.path.join(DRIVE_ROOT, 'problem2_deliverables')
os.makedirs(OUTPUT_DIR, exist_ok=True)

pipeline = BengaliSubtitlePipeline(
    drive_root  = DRIVE_ROOT,
    output_dir  = OUTPUT_DIR,
    device      = DEVICE,
    use_pyannote= False,  # set True + hf_token if pyannote accepted
    hf_token    = None,   # your HF token for pyannote (optional)
)

results = pipeline.run(
    video_path       = TARGET_VIDEO,
    show_name_hint   = 'feluda',   # used for cast registry lookup
    max_duration_sec = 120.0,      # limit to 2min for fast test (set None for full)
    num_speakers     = 3,          # hint to diarizer (Feluda + Topshe + Jatayu)
)

print('\n📦 Deliverables:')
for k, v in results.items():
    print(f'  {k:40s}: {v}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 6 — Preview Subtitle Tracks
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('─' * 60)
print('  Bengali CC (WebVTT) — first 20 lines')
print('─' * 60)
with open(results['vtt_bn'], encoding='utf-8') as f:
    print(''.join(f.readlines()[:20]))

print('─' * 60)
print('  English Subtitles (SRT) — first 20 lines')
print('─' * 60)
with open(results['srt_en'], encoding='utf-8') as f:
    print(''.join(f.readlines()[:20]))

print('─' * 60)
print('  Hindi Subtitles (SRT) — first 20 lines')
print('─' * 60)
with open(results['srt_hi'], encoding='utf-8') as f:
    print(''.join(f.readlines()[:20]))

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 7 — QC Dashboard (Interactive HTML in notebook)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from IPython.display import display, HTML

print(f'  Compliance Score        : {results["compliance_score"]}%')
print(f'  Pass Rate               : {results["pass_rate_pct"]}%')
print(f'  Total Cues              : {results["total_cues"]}')
print(f'  Flagged for Review      : {results["flagged_cues"]}')
print(f'  Hallucinations Suppressed: {results["hallucination_flags_suppressed"]}')
print(f'  ASR Mode Used           : {results["asr_mode"]}')

print('\n🖥️  Loading interactive QC Dashboard...')
with open(results['qc_html'], encoding='utf-8') as f:
    display(HTML(f.read()))

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 8 — Quick Compliance Check vs Industry Specs
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import json

with open(results['qc_json'], encoding='utf-8') as f:
    qc_data = json.load(f)

summary = qc_data['summary']
queue   = qc_data['ranked_review_queue']

print('=' * 60)
print('  Industry Compliance Summary')
print('=' * 60)
print(f'  Overall Compliance Score : {summary["overall_compliance_score"]}%')
print(f'  CPS Violations           : {summary["cps_violations_count"]}')
print(f'  Shot Straddle Events     : {summary["shot_straddles_count"]}')
print(f'  Hallucination Flags      : {summary["hallucination_risks_count"]}')
print(f'  Average CPS              : {summary["average_cps"]}')

if queue:
    print(f'\n  Top 5 Issues for Human Review:')
    for item in queue[:5]:
        print(f'    Rank #{item["rank"]:02d}  |  '
              f'Cue {item["cue_id"]:03d}  |  '
              f'Risk {item["risk_score"]:.2f}  |  '
              f'{item["recommended_action"]}')
        print(f'            → "{item["text"][:70]}"')
else:
    print('\n  ✅ Zero issues flagged — track is broadcast ready!')
print('=' * 60)